In [ ]:
#-------------------------------------------------------------------------------------------#
# Data criacao: 15/11/2025                                                                  #
# Objetivo:                                                                                 #
#   Alerta de ordens pendentes criação de KPI                                               #              
#   ordens pendentes, quantidade de itens e informações de clientes para envio de alerta por#
#   email e telefone de quem tem o cadastro                                                 #
#   Origem camada silver: customer                                                          #
#   Destino camada gold: orders_penging                                                     #
#-------------------------------------------------------------------------------------------#
# 1. Criar uma tabela temporária com os dados da tabela silver                              #
# 2. Criar uma tabela gold com os dados da tabela silver                                    #
# 4. Importar as funções desejadas                                                          #
#-------------------------------------------------------------------------------------------#

In [ ]:
import os
import sys

# Procura automaticamente em qual pasta acima está a 'funcoes_diversas'
caminho = os.getcwd()
while caminho != os.path.dirname(caminho):
    if os.path.exists(os.path.join(caminho, "funcoes_diversas")):
        if caminho not in sys.path:
            sys.path.insert(0, caminho)
        break
    caminho = os.path.dirname(caminho)

# 4. Importa as funções desejadas
from funcoes_diversas.funcoes_util import buscar_caminho, gerar_tabela_temporaria

In [ ]:
mapa_criado = gerar_tabela_temporaria(["customers","orders"],"silver")

In [ ]:
# MAGIC %sql
# MAGIC describe tmp_silver_orders

In [ ]:
df_orders_pending_gold = spark.sql("""
    with pending as(
    select ORD.customer_id,
           ORD.order_date,
           ORD.store_name,
           ORD.status_descricao,
           sum(ORD.quantity) as Quantidade
    from tmp_silver_orders as ORD
    where 1 = 1
      and UPPER(ORD.status_descricao) = 'PENDING'
      group by ORD.customer_id,
           ORD.order_date,
           ORD.store_name,
           ORD.status_descricao)
    select ORD.customer_id,
           ORD.order_date,
           ORD.store_name,
           ORD.status_descricao,
           ORD.Quantidade,
           CUS.first_name,
            CUS.email,
           CUS.phone          
    FROM pending as ORD
    INNER JOIN tmp_silver_customers as CUS
           ON ORD.customer_id = CUS.customer_id
  """)

In [ ]:
gold_path = buscar_caminho("gold")
# transformando consulta em uma tabela Silver 
df_orders_pending_gold.write\
    .mode("overwrite")\
    .format("delta")\
    .option("overwriteSchema", "true")\
    .save(f"{gold_path}/orders_pending")